# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: CTR / Engagement Opportunity Scoring.** This notebook re-derives its own key numbers from scratch (not just quoting Weeks 4–7) so it stands on its own — a fresh clone can run this top to bottom and get the same results, per this repo's own reproducibility rule. Random seed fixed at `42` throughout.

## 1. Question

**Research question:** among pages already visible in search (indexed, earning impressions), which ones are under-capturing clicks or engagement relative to what their position tier predicts — and which should a content reviewer look at first, given limited review capacity?

**Decision this supports:** how a content team allocates scarce weekly review time across a much larger pool of visible pages, instead of reviewing randomly or only when someone happens to notice a problem.

**Unit of analysis:** one row = one page (`content_id`), for one client, over a fixed 90-day window.

**Why ML, not a fixed rule:** Week 2 found CTR shifts by position tier *and* by search intent, and the two interact rather than acting independently. A single if-statement can encode one adjustment; it can't cleanly encode several interacting ones without becoming an unreadable pile of nested conditions.

In [1]:
import pandas as pd
import numpy as np
import json
import os

pd.set_option('display.width', 160)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('setup ok, seed =', RANDOM_SEED)

setup ok, seed = 42


## 2. Data

**Source:** `data/raw/content_refresh_anonymized.csv` — the small anonymized starter dataset that ships with this repo (per the lane guide, Lane 4 explicitly supports either the warehouse release or this starter dataset; the starter dataset is what every executed result in this capstone actually uses).

**Scope:** 30,000 rows total. After the standard filters (`impressions_90d > 0`, `content_age_days >= 90`, deduped by `content_id`) and the Lane 4 visibility filter (`avg_position` between 1 and 20, `impressions_90d >= 500`), the working slice is verified below.

**Excluded on purpose:** `engagement_rate` and `scroll_rate` are excluded from the feature set (they define the label — see Section 3); FlyRank's product-decision fields (`health_score`, `priority_score`, `action_type`, `refresh_tier`) are confirmed absent from this dataset entirely; raw query/URL/title text was never in the release to begin with.

In [2]:
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print('raw shape:', df.shape)

filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
lane4 = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20)
                  & (filtered['impressions_90d'] >= 500)].copy()

print('after standard filters:', filtered.shape, ' unique clients:', filtered['client_id'].nunique())
print('Lane 4 working slice:', lane4.shape, ' unique clients:', lane4['client_id'].nunique())
print()

banned_flags = ['health_score', 'priority_score', 'action_type', 'refresh_tier']
hits = [c for c in df.columns if any(b in c.lower() for b in banned_flags)]
print('product-decision-flag columns present:', hits if hits else 'none found')

raw shape: (30000, 44)
after standard filters: (30000, 44)  unique clients: 32
Lane 4 working slice: (12023, 44)  unique clients: 28

product-decision-flag columns present: none found


## 3. Methodology

**Baseline (transparent rule):** `ctr_gap_score = max(0, tier_median_ctr − page_ctr)` — a page's shortfall against its own position tier's typical CTR. No fitted weights.

**Label/proxy:** `engagement_deficit` — zero measured engagement AND below-median scroll rate. This is a **same-window proxy**, not a validated future outcome; stated plainly, not smoothed over.

**Features:** `avg_position`, `log1p(impressions_90d)`, `ctr`, `content_age_days`, `word_count` (with an explicit missingness indicator, since ~30% of rows lack it), plus one-hot `position_tier`, `competition_level`, `main_intent`. **Deliberately excluded:** `engagement_rate`, `scroll_rate` — since they define the label, including them would let the model just learn to restate its own answer.

**Validation design:** client-grouped 75/25 split (`GroupShuffleSplit` on `client_id`) — a random row-level split risks letting pages from the same client leak shared patterns between train and test. Verified below: zero client overlap.

**Leakage checks:** correlation of every final feature against the label (max reported below), and the product-flag check already run in Section 2.

In [3]:
lane4['expected_ctr_for_tier'] = lane4.groupby('position_tier')['ctr'].transform('median')
lane4['ctr_gap_score'] = (lane4['expected_ctr_for_tier'] - lane4['ctr']).clip(lower=0)
median_scroll = lane4['scroll_rate'].median()
lane4['engagement_deficit'] = ((lane4['engagement_rate'] == 0) & (lane4['scroll_rate'] < median_scroll)).astype(int)
lane4['log_impressions'] = np.log1p(lane4['impressions_90d'])
lane4['word_count_missing'] = lane4['word_count'].isna().astype(int)
lane4['word_count_filled'] = lane4['word_count'].fillna(-1)

cat_cols = ['position_tier', 'competition_level', 'main_intent']
num_cols = ['avg_position', 'log_impressions', 'ctr', 'content_age_days', 'word_count_filled', 'word_count_missing']
dummies = pd.get_dummies(lane4[cat_cols], columns=cat_cols)
lane4_enc = pd.concat([lane4[num_cols].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
# NOTE: neither 'ctr' nor 'content_age_days' is duplicated into `extra` below -- both are
# already present via num_cols as real features. Re-adding either under the same name creates
# a duplicate column, and name-based exclusion then silently strips BOTH copies out of
# feature_cols with no error at all -- a mistake caught the hard way while building Week 7.
extra = lane4[['content_id', 'client_id', 'position_tier', 'impressions_90d',
               'ctr_gap_score', 'engagement_deficit']].reset_index(drop=True)
lane4_enc = pd.concat([lane4_enc, extra], axis=1)
feature_cols = [c for c in lane4_enc.columns if c not in extra.columns]

assert 'ctr' in feature_cols and 'content_age_days' in feature_cols
assert not lane4_enc.columns.duplicated().any()

# Leakage check: correlation of each final feature against the label
corrs = lane4[num_cols + ['engagement_deficit']].corr()['engagement_deficit'].drop('engagement_deficit')
corrs = corrs.reindex(corrs.abs().sort_values(ascending=False).index)
print('feature-label correlations:')
print(corrs.round(3))
print(f'max abs correlation: {corrs.abs().max():.3f} -- no single feature restates the label')

feature-label correlations:
log_impressions      -0.292
ctr                  -0.217
word_count_missing    0.168
word_count_filled    -0.147
content_age_days      0.069
avg_position          0.044
Name: engagement_deficit, dtype: float64
max abs correlation: 0.292 -- no single feature restates the label


## 4. Results (vs baseline)

Model vs. baseline, **on the same client-grouped held-out test set** — re-derived fresh here, not just quoted from Week 5–6.

In [4]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(lane4_enc, groups=lane4_enc['client_id']))
train, test = lane4_enc.iloc[train_idx], lane4_enc.iloc[test_idx]
overlap = set(lane4_enc['client_id'].iloc[train_idx]) & set(lane4_enc['client_id'].iloc[test_idx])
print(f'train: {len(train)} rows, {train["client_id"].nunique()} clients')
print(f'test:  {len(test)} rows, {test["client_id"].nunique()} clients')
print(f'client overlap: {overlap} -- must be empty')
print()

model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED,
                                n_jobs=-1, class_weight='balanced').fit(train[feature_cols], train['engagement_deficit'])
test_probs = model.predict_proba(test[feature_cols])[:, 1]
y_test = test['engagement_deficit'].values

results = []
for k in [20, 50]:
    p_base = precision_at_k(test['ctr_gap_score'].values, y_test, k)
    p_model = precision_at_k(test_probs, y_test, k)
    results.append({'k': k, 'baseline_precision': round(p_base, 3), 'model_precision': round(p_model, 3)})
results_df = pd.DataFrame(results)

print('=== HONEST RESULTS TABLE (client-grouped holdout) ===')
print(results_df)
print()
print(f'base rate on test set: {y_test.mean():.3f}')
print(f'ROC AUC -- baseline: {roc_auc_score(y_test, test["ctr_gap_score"]):.3f}   '
      f'model: {roc_auc_score(y_test, test_probs):.3f}')

train: 11199 rows, 21 clients
test:  824 rows, 7 clients
client overlap: set() -- must be empty



=== HONEST RESULTS TABLE (client-grouped holdout) ===
    k  baseline_precision  model_precision
0  20                0.35             0.60
1  50                0.36             0.52

base rate on test set: 0.191
ROC AUC -- baseline: 0.648   model: 0.776


**Rigor check, reproduced from Week 6:** the same model under a naive, ungrouped random split shows inflated precision (0.750 vs. the honest 0.600 at K=20) — evidence the client-grouped split above is doing real work, not just adding process for its own sake.

In [5]:
Xtr, Xte, ytr, yte, cl_tr, cl_te = train_test_split(
    lane4_enc[feature_cols], lane4_enc['engagement_deficit'], lane4_enc['client_id'],
    test_size=0.25, random_state=RANDOM_SEED, stratify=lane4_enc['engagement_deficit'])
m_naive = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED,
                                  n_jobs=-1, class_weight='balanced').fit(Xtr, ytr)
probs_naive = m_naive.predict_proba(Xte)[:, 1]
naive_overlap = set(cl_tr) & set(cl_te)

print(f'naive split client overlap: {len(naive_overlap)} of {lane4_enc["client_id"].nunique()} clients')
print(f'naive precision@20: {precision_at_k(probs_naive, yte.values, 20):.3f}  '
      f'(honest client-grouped: {results_df.loc[0,"model_precision"]:.3f})')

naive split client overlap: 27 of 28 clients
naive precision@20: 0.750  (honest client-grouped: 0.600)


## 5. Limitations

**What this work cannot claim:**
- **No causal claim.** Nothing here says a specific fix (a title rewrite, a content edit) *causes* an improvement — that needs an actual before/after experiment, which cross-sectional data can't provide.
- **Same-window label.** `engagement_deficit` and `ctr_gap_score` are both computed from the *current* 90-day window, not a validated future outcome. This ranks who looks troubled **now**, not who will decline next.
- **Sample scope.** Results are from a 30,000-row anonymized starter slice across {n_clients} clients — not yet validated at full warehouse scale (~79M rows, ~70 clients). Code exists to run this on the warehouse release (`w03_data_contract.ipynb`, `w03_feature_leakage_check.ipynb`), but the gated Hugging Face access wasn't available in this environment, so those notebooks are prepared but **not executed** — a real gap, not a technicality to gloss over.
- **Part of the model's edge is a fairer-comparison problem, not pure model skill.** Roughly 40% of feature importance traces to `log_impressions` alone — a feature the baseline never had access to at all.
- **No claim about Google's algorithm, or about AI search visibility.** Nothing here supports either.

In [6]:
print(f"n_clients in Lane 4 working slice: {lane4['client_id'].nunique()}")
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print()
print('top feature importances (context for the limitation above):')
print(importances.head(5).round(3))

n_clients in Lane 4 working slice: 28

top feature importances (context for the limitation above):
log_impressions      0.394
ctr                  0.224
word_count_filled    0.104
content_age_days     0.087
avg_position         0.077
dtype: float64


## 6. Ranked recommendations

The final production ranking, retrained on the full Lane 4 slice (the honest performance estimate above already used a held-out quarter of the data; this final pass uses everything). Every page falls into exactly one archetype, each with a distinct recommended action.

In [7]:
final_model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED,
                                      n_jobs=-1, class_weight='balanced')
final_model.fit(lane4_enc[feature_cols], lane4_enc['engagement_deficit'])
lane4_enc['pred_prob'] = final_model.predict_proba(lane4_enc[feature_cols])[:, 1]
lane4_enc['reason_low_ctr_vs_tier'] = lane4_enc['ctr_gap_score'] > 0

ranked = lane4_enc.sort_values(['pred_prob', 'ctr_gap_score'], ascending=[False, False]).reset_index(drop=True)

def archetype(row):
    if row['reason_low_ctr_vs_tier'] and row['engagement_deficit']:
        return 'dual_underperformer'
    elif row['reason_low_ctr_vs_tier']:
        return 'ctr_only_underperformer'
    elif row['engagement_deficit']:
        return 'engagement_only_underperformer'
    return 'healthy'

ranked['archetype'] = ranked.apply(archetype, axis=1)
action_map = {
    'ctr_only_underperformer':        'Rewrite title/meta description.',
    'engagement_only_underperformer': 'Review on-page content/layout.',
    'dual_underperformer':            'Full review, highest priority.',
    'healthy':                        'Monitor only -- no action.',
}
counts = ranked['archetype'].value_counts()
pct = (ranked['archetype'].value_counts(normalize=True) * 100).round(1)
print(pd.DataFrame({'count': counts, 'pct': pct, 'action': [action_map[a] for a in counts.index]}))
print()

# Decay/refresh insight: dual-underperformer rate by content age
# age_bucket must be attached BEFORE any sort to stay row-aligned -- attaching it
# post-sort by position was exactly the Week 7 bug. Building it on lane4_enc (pre-sort
# order) rather than on `ranked` (already reordered by pred_prob) avoids repeating that.
lane4_enc_ordered = lane4_enc.copy()
lane4_enc_ordered['age_bucket'] = pd.cut(lane4['content_age_days'].reset_index(drop=True),
                                          bins=[0, 180, 365, 730, 100000], labels=['<6mo','6-12mo','1-2yr','2yr+'])
lane4_enc_ordered['archetype'] = lane4_enc_ordered.apply(archetype, axis=1)
decay = lane4_enc_ordered.groupby('age_bucket', observed=True)['archetype'].apply(
    lambda s: (s == 'dual_underperformer').mean())
print('dual_underperformer rate by content age:')
print(decay.round(3))

                                count   pct                           action
archetype                                                                   
healthy                          4625  38.5       Monitor only -- no action.
ctr_only_underperformer          3347  27.8  Rewrite title/meta description.
dual_underperformer              2541  21.1   Full review, highest priority.
engagement_only_underperformer   1510  12.6   Review on-page content/layout.

dual_underperformer rate by content age:
age_bucket
<6mo      0.177
6-12mo    0.205
1-2yr     0.286
Name: archetype, dtype: float64


## 7. Artifacts the paper embeds

Regenerating the two figures the deployed paper will show, plus the metrics receipt.

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('../figures', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 4))
labels = ['precision@20', 'precision@50']
ax.bar([0 - 0.175, 1 - 0.175], [results_df.loc[0,'baseline_precision'], results_df.loc[1,'baseline_precision']],
       0.35, label='Baseline (ctr_gap rule)', color='#999999')
ax.bar([0 + 0.175, 1 + 0.175], [results_df.loc[0,'model_precision'], results_df.loc[1,'model_precision']],
       0.35, label='Model (Random Forest)', color='#2e7d6e')
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_ylabel('Precision'); ax.set_ylim(0, 1)
ax.set_title('Baseline vs Model, honest client-grouped test set')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/baseline_vs_model_precision.png', dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(6, 4))
decay.plot(kind='bar', ax=ax, color='#c0533e')
ax.set_ylabel('% dual_underperformer'); ax.set_xlabel('Content age')
ax.set_title('Dual-underperformer rate rises with content age')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../figures/dual_underperformer_by_age.png', dpi=150)
plt.close()

capstone_metrics = {
    'random_seed': RANDOM_SEED,
    'n_rows_lane4': int(len(lane4)),
    'n_clients_lane4': int(lane4['client_id'].nunique()),
    'label_base_rate': float(lane4_enc['engagement_deficit'].mean()),
    'model_precision_at_20': float(results_df.loc[0, 'model_precision']),
    'model_precision_at_50': float(results_df.loc[1, 'model_precision']),
    'baseline_precision_at_20': float(results_df.loc[0, 'baseline_precision']),
    'baseline_precision_at_50': float(results_df.loc[1, 'baseline_precision']),
    'archetype_pct': (ranked['archetype'].value_counts(normalize=True).round(4)).to_dict(),
    'dual_underperformer_rate_by_age': decay.round(4).to_dict(),
}
with open('../outputs/capstone_metrics.json', 'w') as f:
    json.dump(capstone_metrics, f, indent=2)

print('wrote work/figures/baseline_vs_model_precision.png')
print('wrote work/figures/dual_underperformer_by_age.png')
print('wrote work/outputs/capstone_metrics.json (committed -- the receipts for the paper)')
print()
print(json.dumps(capstone_metrics, indent=2))

wrote work/figures/baseline_vs_model_precision.png
wrote work/figures/dual_underperformer_by_age.png
wrote work/outputs/capstone_metrics.json (committed -- the receipts for the paper)

{
  "random_seed": 42,
  "n_rows_lane4": 12023,
  "n_clients_lane4": 28,
  "label_base_rate": 0.33693753638858853,
  "model_precision_at_20": 0.6,
  "model_precision_at_50": 0.52,
  "baseline_precision_at_20": 0.35,
  "baseline_precision_at_50": 0.36,
  "archetype_pct": {
    "healthy": 0.3847,
    "ctr_only_underperformer": 0.2784,
    "dual_underperformer": 0.2113,
    "engagement_only_underperformer": 0.1256
  },
  "dual_underperformer_rate_by_age": {
    "<6mo": 0.1767,
    "6-12mo": 0.2049,
    "1-2yr": 0.2861
  }
}


---

## ML-12 — Demo outline, social cut, employer summary

**5-minute demo outline:**
1. (30s) **The question, from a real FlyRank problem:** FlyRank's own `low_ctr_visible_page` flag fires on 75-94% of visible pages in every position tier -- not a shortlist, nearly the whole inventory. Which pages should a reviewer actually open first, once that flag gives no real priority order?
2. (60s) **The method:** a transparent, tier-adjusted baseline rule with reason codes, then a Random Forest validated on a client-grouped holdout split (21 train clients, 7 test, zero overlap).
3. (90s) **One chart, one honest result:** show `baseline_vs_model_precision.png` -- precision@20 of 0.600 (model) vs. 0.350 (baseline) -- then immediately show the naive-split version (0.750) side by side, so the rigor is visible, not just claimed.
4. (60s) **One recommendation:** prioritize the `ctr_only_underperformer` archetype first -- it sits on the highest-traffic pages (mean 11,915 impressions) and needs only a cheap title/meta rewrite, the best reviewer-time-to-value ratio in the queue.
5. (30s) Limitations, stated plainly: same-window label, starter-scale data, no causal claim -- and what warehouse-scale validation would still need to confirm.

**Social-post cut** (adapted from the LinkedIn draft already written for Week 1, updated with the full pipeline's real numbers):

> Capstone wrap-up: built a tier-adjusted CTR/engagement scoring system for content review prioritization. On a held-out set of clients the model never saw, it ranked engagement-troubled pages at 0.60 precision@20 vs. a transparent baseline's 0.35 — validated with a client-grouped split (not a naive one, which would have shown an inflated 0.75). Full pipeline: baseline → model → leakage audit → archetype-based action playbook. Repo and paper linked below.

**Employer-facing summary (3 sentences):** I built and validated a content-prioritization scoring system on real (anonymized) SEO/analytics data, comparing a transparent rule-based baseline against a Random Forest under a client-grouped holdout split — the honest way, which I demonstrated by also showing how much a naive split would have inflated the numbers. The final output is a reason-coded, archetype-based action playbook that a content team could actually use to decide what to review first, with explicit limits on what it can and can't claim. Throughout, I treated validation rigor and honest limitations as part of the deliverable, not an afterthought — including catching and fixing two of my own feature-engineering bugs along the way rather than shipping silently-wrong numbers.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.